# Crisis Negotiator — GRPO Training

Trains **Qwen2.5-3B-Instruct** with TRL GRPO on a 6-agent crisis negotiation RL environment.

| | |
|---|---|
| **Env** | 6 agents, hidden state, deception, partial observability |
| **Model** | Qwen2.5-3B-Instruct + LoRA r=16 (fp16) |
| **Training** | 32 prompts x 4 generations, ~10 min on T4 |

**Colab: Runtime > Change runtime type > T4 GPU**

In [ ]:
# Cell 1 — Install
!pip install -q "openenv-core>=0.2.3" "trl>=0.14" peft accelerate datasets sentence-transformers matplotlib
print("Installed")

In [ ]:
# Cell 2 — Clone repo
import pathlib, subprocess, shutil, os, sys
if not pathlib.Path("server").exists():
    subprocess.run(["git","clone","-b","kaggle_version",
                    "https://github.com/Dinesh052/Test.git","_r"], check=True)
    for item in pathlib.Path("_r").iterdir():
        dest = pathlib.Path(item.name)
        if not dest.exists(): item.rename(dest)
    shutil.rmtree("_r", ignore_errors=True)
sys.path.insert(0, ".")
print("Repo ready")

In [ ]:
# Cell 3 — GPU check
import torch
assert torch.cuda.is_available(), "No GPU!"
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | {p.total_memory/1e9:.1f} GB")

In [ ]:
# Cell 4 — Smoke-test environment
from server.environment import CrisisNegotiatorEnvironment
from models import NegotiatorAction

env = CrisisNegotiatorEnvironment()
obs = env.reset(task_id="easy_domestic_desperate", seed=42)
print(f"Reset OK | phase={obs.phase}")
obs = env.step(NegotiatorAction(action_type="emotional_label",
    content="It sounds like you are feeling overwhelmed.",
    reasoning="empathy", target="hostage_taker"))
print(f"Step OK  | reward={obs.reward:.4f} done={obs.done}")

In [ ]:
# Cell 5 — Load model + LoRA
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map="auto", trust_remote_code=True, torch_dtype=torch.float16)
lora = LoraConfig(r=16, lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.0, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# Cell 6 — Build training prompts
from datasets import Dataset

SYSTEM = """You are an FBI-trained crisis negotiator. Respond with ONE JSON:
{"action_type": "<type>", "content": "<words>", "reasoning": "<strategy>",
 "target": "hostage_taker", "belief_agitation": <0-10>, "belief_lying": <bool>}
Action types: emotional_label, mirror, open_question, acknowledge_demand,
offer_concession, buy_time, speak, push_back_commander, request_demand, ask_proof_of_life"""

HEUR = [
    ("emotional_label", "It sounds like you are carrying a lot of pain."),
    ("mirror", "You said something important."),
    ("open_question", "What happened from your side?"),
    ("acknowledge_demand", "I hear what you need. Let me work on that."),
]
N = 32  # increase to 64+ for better results
DIFFS = ["easy","medium","hard"]
rows = []
for i in range(N):
    env = CrisisNegotiatorEnvironment()
    diff = DIFFS[i % 3]
    obs = env.reset(task_id=f"generate:{diff}", seed=42+i)
    for s in range(i % 4):
        at, ct = HEUR[s % len(HEUR)]
        obs = env.step(NegotiatorAction(action_type=at, content=ct, reasoning="pre", target="hostage_taker"))
        if obs.done: break
    parts = [f"Scenario: {obs.scenario_brief}", f"Step {obs.step}, patience={obs.commander_patience}"]
    if obs.stated_demands:
        parts.append("Demands: " + "; ".join(d["text"] for d in obs.stated_demands))
    for e in (obs.dialogue_history or [])[-4:]:
        parts.append(f"  {e['speaker'][:3].upper()}: {e['content'][:120]}")
    parts.append("Respond with one JSON:")
    msgs = [{"role":"system","content":SYSTEM},{"role":"user","content":"\n".join(parts)}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    rows.append({"prompt": text, "prompt_idx": i})
dataset = Dataset.from_list(rows)
print(f"Built {len(dataset)} prompts")

In [ ]:
# Cell 7 — Reward function
import json, re

VALID = {"speak","request_demand","acknowledge_demand","offer_concession",
    "ask_proof_of_life","buy_time","push_back_commander","emotional_label","mirror","open_question"}

def _parse(text):
    text = re.sub(r"```(?:json)?\s*", "", text.strip())
    text = re.sub(r"\s*```\s*$", "", text).strip()
    try:
        d = json.loads(text)
        if isinstance(d, dict): return d
    except: pass
    m = re.search(r'\{[^{}]*"action_type"[^{}]*\}', text, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    return None

def reward_fn(completions, **kwargs):
    scores = []
    for c in completions:
        p = _parse(c)
        if not p: scores.append(-0.3); continue
        at = p.get("action_type","speak")
        content = p.get("content","")
        if at not in VALID: scores.append(-0.2); continue
        s = 0.05
        if at in ("emotional_label","mirror","open_question"): s += 0.15
        elif at in ("acknowledge_demand","offer_concession"): s += 0.10
        if at == "acknowledge_demand": s += 0.10
        if p.get("belief_agitation") is not None: s += 0.05
        if p.get("belief_lying") is not None: s += 0.03
        if len(p.get("reasoning","")) > 20: s += 0.05
        if any(w in content.lower() for w in ["kill","force","breach"]): s -= 0.20
        if len(content) < 10: s -= 0.10
        scores.append(max(-0.5, min(0.5, s)))
    return scores

print(f"Test: {reward_fn([json.dumps({'action_type':'emotional_label','content':'I hear you','reasoning':'empathy rapport build','belief_agitation':7}), 'bad'])}")

In [ ]:
# Cell 8 — Train with GRPO
import time
from trl import GRPOConfig, GRPOTrainer

cfg = GRPOConfig(
    output_dir="./crisis-negotiator-colab",
    num_generations=4, generation_batch_size=4,
    max_completion_length=192, temperature=0.9, beta=0.04,
    per_device_train_batch_size=4, num_train_epochs=1,
    learning_rate=5e-6, logging_steps=4,
    save_steps=9999, save_total_limit=1,
    fp16=True, gradient_checkpointing=False,
    report_to=[], remove_unused_columns=False,
)
trainer = GRPOTrainer(model=model, args=cfg,
    reward_funcs=[reward_fn], train_dataset=dataset,
    processing_class=tokenizer)

t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
model.save_pretrained("./crisis-negotiator-colab")
tokenizer.save_pretrained("./crisis-negotiator-colab")
print(f"\nDone in {elapsed:.1f} min")

## Evaluation

In [ ]:
# Cell 9 — Evaluate random vs heuristic
import random

ACTIONS = list(VALID)
HEUR_CYCLE = [
    ("emotional_label","It sounds like you are feeling scared and overwhelmed."),
    ("mirror","You said you just want to be heard."),
    ("open_question","Tell me more about what brought you here?"),
    ("acknowledge_demand","I hear your demand and want to work on it."),
    ("emotional_label","It must be frustrating to feel unheard."),
    ("buy_time","Let us take a moment, there is no rush."),
    ("acknowledge_demand","Your concern is completely valid."),
    ("offer_concession","I am going to work on getting what you need."),
    ("mirror","You mentioned wanting to see your kids."),
    ("open_question","How can we keep everyone safe?"),
]
SCENARIOS = ["easy_domestic_desperate","easy_bank_surrender","easy_workplace_grievance",
    "medium_custody_ideologue","medium_pharmacy_calculated","medium_bridge_unstable","medium_protest_drift",
    "hard_embassy_calculated","hard_hospital_bluffer","hard_school_unstable_drift","hard_compound_ideologue"]

def run_eval(name, act_fn, n=30):
    res = []
    for t in range(n):
        env = CrisisNegotiatorEnvironment()
        obs = env.reset(task_id=SCENARIOS[t%len(SCENARIOS)], seed=10000+t)
        for si in range(25):
            if obs.done: break
            at, ct = act_fn(obs, si, t)
            obs = env.step(NegotiatorAction(action_type=at, content=ct, reasoning=name, target="hostage_taker"))
        out = obs.message.split(":")[1].strip().split(".")[0] if ":" in obs.message else "?"
        res.append({"reward":obs.reward,"outcome":out})
    rr = [r["reward"] for r in res]
    sur = sum(1 for r in res if "surrender" in r["outcome"] or "released" in r["outcome"])
    harm = sum(1 for r in res if "harm" in r["outcome"])
    print(f"  {name:12s} | reward={sum(rr)/len(rr):.3f} | surrender={sur/n*100:.0f}% | harm={harm/n*100:.0f}%")
    return res

print("Policy         | Reward  | Surrender | Harm")
print("-"*55)
r_rand = run_eval("random", lambda o,s,t: (random.Random(10000+t*100+s).choice(ACTIONS), "I want to help."))
r_heur = run_eval("heuristic", lambda o,s,t: HEUR_CYCLE[s%len(HEUR_CYCLE)])

In [ ]:
# Cell 10 — Plot
import matplotlib.pyplot as plt, numpy as np

fig, ax = plt.subplots(figsize=(8,4))
policies = ["Random","Heuristic"]
means = [np.mean([r["reward"] for r in r_rand]), np.mean([r["reward"] for r in r_heur])]
colors = ["#C44E52","#55A868"]
bars = ax.bar(policies, means, color=colors, width=0.4)
for b,v in zip(bars,means): ax.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.3f}", ha="center")
ax.set_ylabel("Mean Reward"); ax.set_ylim(0,1.05)
ax.set_title("Crisis Negotiator — Policy Comparison (n=30)")
plt.tight_layout(); plt.savefig("reward_curve_colab.png", dpi=150); plt.show()
print("Saved reward_curve_colab.png")

In [ ]:
# Cell 11 — Summary
import pathlib
adapter = pathlib.Path("./crisis-negotiator-colab")
if adapter.exists():
    total = sum(f.stat().st_size for f in adapter.glob("*") if f.is_file())/1e6
    print(f"Adapter: {total:.1f} MB")
print(f"Random:    {np.mean([r['reward'] for r in r_rand]):.3f}")
print(f"Heuristic: {np.mean([r['reward'] for r in r_heur]):.3f}")
print("\nDone! Adapter in ./crisis-negotiator-colab/")